In [89]:
with open("The Adventures of Sherlock Holmes.txt", 'r') as text_file:
    text = text_file.read()[1474:] # remove metadata

In [90]:
pre_bpe_vocab_size = len(set(text))

In [91]:
joined = "".join(text)
text_utf_8 = list(joined.encode("utf-8"))
pre_bpe_text_len = len(text_utf_8)
pre_bpe_text_len

593820

# Building token vocab

In [92]:
from collections import Counter
from itertools import pairwise


def get_pair_counter(input_seq):
    return Counter(pairwise(input_seq))

In [93]:
def top_pair(pair_counter):
    return max(pair_counter, key=pair_counter.get)

In [94]:
def replace_pair(input_seq, old_pair, new_elem):
    def _pair(i):
        return (input_seq[i], input_seq[i + 1])

    result = []
    i = 0
    while i < len(input_seq):
        if i < len(input_seq) - 1 and _pair(i) == old_pair:
            result.append(new_elem)
            i += 2
        else:
            result.append(input_seq[i])
            i += 1
    return result

In [95]:
vocab_size = 1000

bpe_map: dict[tuple[int, int], int] = {}

bpe_text = list(text_utf_8)

for i in range(256, vocab_size):
    pair_counter = get_pair_counter(bpe_text)
    pair = top_pair(pair_counter)
    bpe_text = replace_pair(bpe_text, pair, i)
    bpe_map[pair] = i

In [96]:
print(f"Original input size: {len(text)}")
print(f"Compressed size: {len(bpe_text)}")
print(f"New vocab size: {vocab_size}")
print(f"Compression ratio: {len(text) / len(bpe_text):.2f}x")

Original input size: 580127
Compressed size: 213973
New vocab size: 1000
Compression ratio: 2.71x


In [97]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (fst, snd), idx in bpe_map.items():
    vocab[idx] = vocab[fst] + vocab[snd]

In [98]:
def encode(input_seq):
    utf = list(input_seq.encode('utf-8'))
    for pair, key in bpe_map.items():
        utf = replace_pair(utf, pair, key)
    return utf

In [99]:
def decode(input_seq):
    tokens = b"".join(vocab[idx] for idx in input_seq)
    return tokens.decode('utf-8', errors='replace')

In [101]:
pre_encode = text[:10000]
pre_encode == decode(encode(pre_encode))

True